# Continuation Method for Pareto Exploration for the problems below

# Problems (2 Objectives)

# Defining Mathematical objectives
Given $C^i \in \R^2 \ \& \  \R^3 $

* # 1

$
\min_{x \in \mathbb{R}^n} F(x) = \min_{x \in \mathbb{R}^n} \left\{ \frac{1}{2}(x-C^i)^T Q^i (x-C^i) \right\}_{i=1}^2
$

In [1]:
import numpy as np
import cvxpy as cp
from scipy.optimize import minimize
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.sparse.linalg import LinearOperator, minres
from source_codes.method import *
#*****problem: 2-varibale, 2-objective*****
from source_codes.x2CQ_problem import x_c_problem


#*****problem: 3-varibale, 2-objective*****
#from source_codes.x3CQ_problem import x_c_problem
from qpsolvers import solve_qp


In [13]:
def beta_basis(m):
        """Construct n x k matrix with row-sum 0 and full row rank (works if n <= k-1)."""
        k = m-1
        if k > m:
            raise ValueError("Impossible: need n <= k-1 for full row rank with row-sum 0.")
        B = np.zeros((k, m), dtype=float)
        for i in range(k):
            B[i, i] = 1.0
            B[i, i+1] = -1.0
        return B

beta_basis = beta_basis(5)  # for 4-objective

np.linalg.matrix_rank(beta_basis)

4

In [14]:
#**********Setting up the problem**********#
#Now we define the hyperparameters and random seeds. In particular, maxiter controls the number of MINRES iterations. Since this synthetic example has a very small Hessian matrix, at most 3 iterations are needed.
s = 0.5 # 0.1            # Step size.
maxiter = 2 #1000         # Maximum allowable number of iterations in MINRES. You can vary it from 1 to 3 to see how results improve with this parameter.
np.random.seed(42)

In [15]:
#problem setting
problem = x_c_problem()
n, m = problem.n, problem.m
# Generate the initial Pareto optimal point.
x0 = problem.sample_pareto_set() #10
f0 = problem.f(x0)
g0 = problem.grad(x0)
print("Initial point in the decision space: ", x0)
print("Initial point in the objective space: ", f0)


Initial point in the decision space:  [3.11216336 2.49465184]
Initial point in the objective space:  [0.20904815 2.35295726]


In [16]:
# Our method
#Next, we solve $\alpha$ from Equation (3) in the paper:
def compute_alpha(g,m=2):
    alpha = cp.Variable(m)
    objective = cp.Minimize(cp.sum_squares(alpha.T @ g))
    constraints = [alpha >= 0, cp.sum(alpha) == 1]
    alpha_prob = cp.Problem(objective, constraints)
    alpha_prob.solve()
    alpha = ndarray(alpha.value).ravel()
    return alpha

alpha = compute_alpha(g0, m= 2)
alpha 

array([0.79591837, 0.20408163])

Computing projected directions influenced by preference vector computed via MINRES

In [17]:
def minres_solver(problem,x0, alpha,b,  reg = 0.0):
    n = len(b)
    x, _ = minres(LinearOperator((n, n), matvec=lambda v: problem.hvp(x0, alpha, v,reg = reg), rmatvec=lambda v: problem.hvp(x0, alpha, v,reg=reg)), b, maxiter=maxiter) #, rmatvec=H_op
    #print(f"MINRES info: {_}")
    return np.array(x)



def beta_basis(m):
    """Construct n x k matrix with row-sum 0 and full row rank (works if n <= k-1)."""
    k = m-1
    if k > m:
        raise ValueError("Impossible: need n <= k-1 for full row rank with row-sum 0.")
    B = np.zeros((k, m), dtype=float)
    for i in range(k):
        B[i, i] = 1.0
        B[i, i+1] = -1.0
    return B

def generate_span_vi(problem, x0, alpha, reg, n_obj=3):
    """
    Generates two orthogonal vectors v1, v2 in the span of beta1, beta2.
    Returns: (v1, v2,...) as a tuple of numpy arrays.
    """

    vi = []
    beta = beta_basis(n_obj)
    for b in beta:
        rhs = problem.grad(x0).T @ b  # shape: (n,)
        v = minres_solver(problem, x0, alpha, rhs, reg= reg)
        v = v if np.linalg.norm(v) == 0 else v/np.linalg.norm(v) 
        vi.append(v)
    # Compute reduced QR of v.T (shape 3 x 2 -> Q: 3x2, R: 2x2)
    vi = np.array(vi)  # shape k x m
    Q, R = np.linalg.qr(vi.T, mode='reduced')

    # Make R have positive diagonal (canonical QR)
    diag_sign = np.sign(np.diag(R))
    diag_sign[diag_sign == 0] = 1.0
    Q = Q * diag_sign  # broadcast to columns

    # Orthonormal rows are the transpose of Q
    v_orth = Q.T  # shape k x m
    return v_orth

def proj_general(V, d):
    V = np.asarray(V, float).T          # (n,k)
    d = np.asarray(d, float).ravel()  # (n,)
    M = V.T @ V              # (k,k)
    #print(V.shape, d.shape, M.shape)     
    if np.allclose(M, np.eye(M.shape[0])):
        return V @ (V.T @ d)  # (n,)  # V.T @ d    (k,)
    return V @ np.linalg.solve(M, V.T @ d)



In [18]:
def compute_descent_direction(problem, x0, alpha, grads, pref, solvers = "minres", reg =0):
    """
    V: (k, n) array of v_j vectors
    grad_f: (m, n) array of gradients ∇f_i(x)
    P: (m,) array of weights P_i
    Returns: optimal descent direction d ∈ ℝ^n
    """
    selected_grads = []
    selected_prefs = []
    for i in range(len(pref)):
        if pref[i] < 0:
            selected_grads.append(grads[i])
            selected_prefs.append(pref[i])

        #elif pref[i] >= 0:
        #    selected_grads.append(np.ones_like(grads[i]))

    if len(selected_grads) >= 1:
        G = np.vstack(selected_grads).T  # (n, k) matrix of selected gradients
        #assert G.shape[1] == len(selected_grads), "Number of selected gradients must match the number of columns in G."     
        

        m = G.shape[1]  # Number of selected gradients
        n = G.shape[0]  # Number of features
        #A = np.ones((1, m))     # sum of theta's = 1
        #b = np.array([1.0])

        D = np.zeros((n, m))
        P = np.matmul(G.T, G)
        #P = 0.5 * (P + P.T)

        for i in range(m):
            q = G.T @ G[:, i] #P[:, i] not P[i, :]
            theta = solve_qp(P, q, G = None, h = None, A =  None, b =  None, lb = np.zeros((m)), ub = None, solver="cvxopt")
            D[:, i] = G[:, i] + np.matmul(G, theta)

    
        d =  D @ np.array(selected_prefs) 
        vi = generate_span_vi(problem, x0, alpha, reg,grads.shape[0])  # Generate orthonormal basis in the span of beta vectors
        proj_d = proj_general(vi, d)

        #print("d", d)
        #print("proj_d", proj_d)
        return proj_d
    
    else:
        d = grads.T @ pref  # If no gradients are selected, use the full gradient
        # Solve for v
        vi = generate_span_vi(problem, x0, alpha, reg,grads.shape[0])  # Generate orthonormal basis in the span of beta vectors
        proj_d = proj_general(vi, d)
        v = proj_d
        return v

### Corrector

In [19]:
## Using MGDA with line search to find a Pareto optimal solution x* and as a corrector step.

gamma = 0.9         # The decay factor in line search.
num_exp = 10        # How many times you want to repeat the experiment with a different seed.
# Hyperparameters for line search.
ths = 1e-5          # For detecting convergence. Smaller value takes longer to converge.
eta_init =  1        # Initial step size in line search.
c1 = 0              # For strong Wolfe condition. Cannot be negative. Typically 1e-4. Larger values requires more iterations in line search.

def line_search(x, f, grad, d, eta, c1):
    x = ndarray(x).ravel()
    assert x.size == n
    f = ndarray(f).ravel()
    assert f.size == m
    grad = ndarray(grad)
    assert grad.shape == (m, n)
    d = ndarray(d).ravel()
    assert d.size == n
    while True:
        x_new = x + eta * d
        f_new = problem.f(x_new)
        if np.all([fi_new <= fi + c1 * gradi.dot(d) * eta for fi, gradi, fi_new in zip(f, grad, f_new)]):
            return eta
        eta *= gamma

def mgda_optimize(x):
    x = ndarray(x).ravel()
    assert x.size == n
    x_iter = np.copy(x)
    while True:
        g_iter = problem.grad(x_iter)
        f_iter = problem.f(x_iter)
        alpha_iter = compute_alpha(g_iter)
        # Negative sign here because d must be a *descent* direction.
        d = -ndarray(alpha_iter.T @ g_iter).ravel()
        # Make sure they are indeed descent.
        for gi in g_iter:
            assert gi.dot(d) <= 0 or np.isclose(gi.dot(d), 0)
        # Termination condition 1: gradient is too small.
        if np.linalg.norm(d) < ths:
            return x_iter
        eta = line_search(x_iter, f_iter, g_iter, d, eta_init, c1)
        x_iter += eta * d
        # Termination condition 2: change is too little. Effectively, this means eta is too small.
        if eta * np.linalg.norm(d) < ths:
            print("change too small, increase step size")
            return x_iter 
        
#mgda_optimize(np.array([2.0,1.5,0.0]))

In [24]:
# Comment out '%matplotlib tk' and uncomment '%matplotlib inline' if this cell does not generate figures on you computer.
%matplotlib tk
#%matplotlib inline

# Collect all the directions we would like to plot: (direction, color, label, s_min, s_max, linewidth)
# We will plot f(x0 + s * normalize(direction)) with s \in [s_min, s_max]
# s =0.5

vi = []
norms = np.linalg.norm(g0, axis=1)
g_normalized = g0 / norms[:, np.newaxis]

K =[np.array([-0.4, 0]),np.array([-0.5, 0]),
    np.array([-0.6, 0]),np.array([-0.7, 0]),np.array([-0.8, 0]),np.array([-0.9, 0])]#,

#for i in range(K):
for i,pref in enumerate(K):
    v= compute_descent_direction(problem, x0, alpha, g_normalized, pref, solvers = "minres", reg =0.1) 
    #v = minres_solver(rhs)
    vi.append((v, 'tab:orange', 'predictor' if i == 0 else None, -s, s, 2))
vi.append((g0[0], 'tab:blue', r'$\nabla f_1(x^*)$', 0, s, 4))
vi.append((g0[1], 'tab:green', r'$\nabla f_2(x^*)$', 0, s, 4))

# Plot the Pareto set.
if n==2:
    fig_ps = plt.figure(figsize=(8, 8))
    ax_ps = fig_ps.add_subplot(1, 1, 1)
    problem.plot_pareto_set(ax_ps)
    ax_ps.scatter(x0[0], x0[1],c='tab:red', s=100)
    for idx, (v, color, label, _, _, _) in enumerate(vi):
        #draw_arrow_3d(ax_ps, x0 + v / np.linalg.norm(v), x0, color)
        if idx == len(vi)-1 or idx ==len(vi)-2:
            new_x = x0 - s*v / np.linalg.norm(v)
            ax_ps.plot([x0[0], new_x[0]], [x0[1], new_x[1]],"-", c=color,label = label,linewidth=2.5)
        else:
            new_x = x0 + s*v / np.linalg.norm(v)
            ax_ps.plot([x0[0], new_x[0]], [x0[1], new_x[1]],"-", c=color,label = label,linewidth=2.5)
            corr= mgda_optimize(new_x) #corrector_step(new_x, L_vec_func = problem.f, G_mat_func = problem.grad, sigma = 0.01, lr=s,h_mode="exact", h_kwargs=None) #mgda_optimize(new_x)
            ax_ps.plot([new_x[0], corr[0]], [new_x[1], corr[1]],"-o", c="blue",linewidth=2)
            if idx ==0:
                ax_ps.plot([new_x[0], corr[0]], [new_x[1], corr[1]],"-o", c="blue",label = "corrector",linewidth=2)

    ax_ps.set_xlabel(r'$x_1$', fontsize = 20)
    ax_ps.set_ylabel(r'$x_2$', fontsize = 20)
    ax_ps.tick_params(axis='x', labelsize=14)
    ax_ps.tick_params(axis='y', labelsize=14)
    ax_ps.legend(fontsize = 14)
    plt.show()

elif n==3:
    fig_ps = plt.figure(figsize=(8, 8))
    ax_ps = fig_ps.add_subplot(1, 1, 1, projection='3d')
    problem.plot_pareto_set(ax_ps)
    ax_ps.scatter(x0[0], x0[1], x0[2], c='tab:red', s=100)
    for idx, (v, color, label, _, _, _) in enumerate(vi):
        #draw_arrow_3d(ax_ps, x0 + v / np.linalg.norm(v), x0, color)
        if idx == len(vi)-1 or idx ==len(vi)-2:
            new_x = x0 - s*v / np.linalg.norm(v)
            ax_ps.plot([x0[0], new_x[0]], [x0[1], new_x[1]], [x0[2], new_x[2]],"-", c=color,label = label,linewidth=2)
        else:
            new_x = x0 + s*v / np.linalg.norm(v)
            ax_ps.plot([x0[0], new_x[0]], [x0[1], new_x[1]], [x0[2], new_x[2]],"-", c=color,label = label,linewidth=2)

    ax_ps.set_xlabel(r'$x_1$')
    ax_ps.set_ylabel(r'$x_2$')
    ax_ps.set_zlabel(r'$x_3$')
    ax_ps.legend()
    plt.show()

# Plot the Pareto front.
fig_pf = plt.figure(figsize=(8, 8))
ax_pf = fig_pf.add_subplot(1, 1, 1)
ax_pf.scatter(f0[0], f0[1], c='tab:red', label='Pareto optimal $f(x^*)$', s=50)
problem.plot_pareto_front(ax_pf, label='Pareto front')
func = []
print("f0",f0)
for idx, (v, color, label, s0, s1, _) in enumerate(vi):
    
    if idx == 0 :
        #fi = [problem.f(x0 - si * v / np.linalg.norm(v)) for si in np.linspace(s0, s1, 21)]
        fi = problem.f(x0 + s * v / np.linalg.norm(v))
        print(fi)
        corr= problem.f(mgda_optimize(x0 + s * v/ np.linalg.norm(v))) #corrector_step(x0 + s * v/ np.linalg.norm(v), L_vec_func = problem.f, G_mat_func = problem.grad, sigma = 0.01, lr=s,h_mode="exact", h_kwargs=None))) #f(mgda_optimize(x0 + s * v/ np.linalg.norm(v) ))
        print("corr", corr)
        ax_pf.plot([fi[0], f0[0]], [fi[ 1], f0[1]], c=color, label=label, linewidth=2.5)
        ax_pf.plot([fi[0], corr[0]], [fi[1], corr[1]],"-o", c="blue",label = "Corrector",linewidth=2)
    elif idx == len(vi)-1 or idx ==len(vi)-2:
        fi = problem.f(x0 - s * v / np.linalg.norm(v))
        ax_pf.plot([fi[0], f0[0]], [fi[1], f0[1]], c=color, label=label, linewidth=2.5) 
    else:
        #fi = [problem.f(x0 - si * v / np.linalg.norm(v)) for si in np.linspace(s0, s1, 21)]
        fi = problem.f(x0 + s * v / np.linalg.norm(v))
        print(fi)
        ax_pf.plot([fi[0], f0[0]], [fi[ 1], f0[1]], c=color, label=label, linewidth=2.5) 
        corr= problem.f(mgda_optimize(x0 + s * v/ np.linalg.norm(v))) #corrector_step(x0 + s * v/ np.linalg.norm(v), L_vec_func = problem.f, G_mat_func = problem.grad, sigma = 0.01, lr=s,h_mode="exact", h_kwargs=None))) #f(mgda_optimize(x0 + s * v/ np.linalg.norm(v) ))
        print("corr", corr)
        ax_pf.plot([fi[0], corr[0]], [fi[1], corr[1]], "-o", c="blue",linewidth=2.5)
        
            
        
        
ax_pf.legend()
plt.show()

ax_pf.set_xlabel(r'$f_1$', fontsize = 20)
ax_pf.set_ylabel(r'$f_2$', fontsize = 20)
ax_pf.tick_params(axis='x', labelsize=14)
ax_pf.tick_params(axis='y', labelsize=14)

ax_pf.legend(fontsize = 14)
plt.show()


f0 [2.43718909 0.21882972]
[1.39784268 0.64954482]
corr [1.39088941 0.64509726]
[1.39784268 0.64954482]
corr [1.39088941 0.64509726]
[1.39784268 0.64954482]
corr [1.39088941 0.64509726]
[1.39784268 0.64954482]
corr [1.39088941 0.64509726]
[1.39784268 0.64954482]
corr [1.39088941 0.64509726]
[1.39784268 0.64954482]
corr [1.39088941 0.64509726]


In [23]:
s = 0.5

In [22]:
# Now generate more results with different random seeds.
#We then repeat the experiment with different random seeds. This figure can be found in the supplemental material. 
#Notice that by moving along the tangent directions, we cover a large portion of the analytic Pareto front. As a comparison, if we walk along the gradients, we quickly diverge from the Pareto front.
vi = []
xi = []
fi = []
# (direction, color, label, s_min, s_max, transparency)
for seed in range(40):
    np.random.seed(seed)
    # Generate x0.
    x0 = problem.sample_pareto_set()
    f0 = problem.f(x0)
    g0 = problem.grad(x0)
    norms = np.linalg.norm(g0, axis=1)
    g_normalized = g0 / norms[:, np.newaxis]
    alpha0 = compute_alpha(g0, m)
    fi.append(f0)
    # Generate 1 direction for each x0.
    pref = -np.random.rand(m)
    v= compute_descent_direction(problem, x0, alpha0, g_normalized, pref, solvers = "minres", reg =0.1) 
    vi.append((v, 'tab:orange', 'MINRES' if seed == 0 else None, -s, s, 0.75, 2))
    xi.append(x0)
    vi.append((g0[0], 'tab:blue', r'$\nabla f_1(x^*)$' if seed == 0 else None, 0, s, 0.1, 4))
    xi.append(x0)
    vi.append((g0[1], 'tab:green', r'$\nabla f_2(x^*)$' if seed == 0 else None, 0, s, 0.1, 4))
    xi.append(x0)
fi = ndarray(fi)

# Plot more results. This is the figure in the supplemental material.
fig_pf = plt.figure(figsize=(5, 5))
ax_pf = fig_pf.add_subplot(1, 1, 1)
problem.plot_pareto_front(ax_pf, label='Pareto front')
for x0, (v, color, label, s0, s1, a, l) in zip(xi, vi):
    if color == 'tab:blue' or color == 'tabe:green':
        fi = [problem.f(x0 - si * v / np.linalg.norm(v)) for si in np.linspace(s0, s1, 21)]

    else:
        fi = [problem.f(x0 + si * v / np.linalg.norm(v)) for si in np.linspace(s0, s1, 21)]
    fi = ndarray(fi)
    ax_pf.plot(fi[:, 0], fi[:, 1], c=color, label=label, linewidth=l, alpha=a)
fi = [problem.f(x1i) for x1i in xi]
fi = ndarray(fi)
ax_pf.scatter(fi[:, 0], fi[:, 1], c='tab:red', label='Pareto optimal $f(x^*)$', s=20, zorder=100)
ax_pf.legend()
plt.show()